# Unit 08 · Putting it together

Everything so far, end to end: two X-rays in, a pelvis mesh out, scored against the CT; then the whole results
story in one honest table, the checklist that says where every number comes from, and the report.

| Part | After video 14 | What you build | Suggested driver |
|---|---|---|---|
| A | the pipeline fly-through | X-rays → network input → occupancy → mesh in mm | Adir / Noa / Adir |
| – | (Claude's) the demo | one test patient: the X-rays, the predicted mesh next to the CT's, the metrics, OBJ files | |
| B | the results story | the best and worst cases, the honest table (mean ± sd, n) | Noa / Adir |
| C | the report | (Claude's) the reproduce-everything checklist, the report outline, the personal-contribution table | both |

**Needs:** units 02–05 moved into `src/xray3d` (`shape.mesh_from_occupancy`, `metrics.evaluate`, `metrics.ce_angle`,
`nets.XrayTo3D`, `losses`, `training`), the processed dataset v1 (unit 02), `splits/split_v1.json` (unit 01), and a
**trained model**: one of unit 06's Kaggle runs (its folder `runs/<name>/` with `best.pt`, downloaded from the Kaggle
notebook's output into `trainer.runs_root()`), or unit 05's `first-2view`. Part B also reads the results files of
units 03 (`results/baselines.csv`), 06 (`results/experiment_v1.csv`) and 07 (`results/sim_vs_real_v1.csv`); whatever
is missing is reported, not fatal. Everything runs on the laptop's CPU in a few minutes.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "08-putting-it-together"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

### Parameters

`MODEL` is the run whose network the demo uses. Put the name of unit 06's best **2-view** run here (the run with the
highest val Dice of the 2-view arm; choose it on **val**, never on test). `DEMO_SUBJECT = None` picks the test subject
with the **median** Dice, so the demo shows a typical case, not the best one.

In [ ]:
MODEL = os.environ.get("XRAY3D_DEMO_MODEL", "first-2view")   # a run name under trainer.runs_root(), or a .pt path
WHICH = "best"            # "best" (highest val Dice) or "last"
DEMO_SUBJECT = None       # e.g. "s0878" (a TEST subject), or None = the test subject with the median Dice

In [ ]:
# (Claude's) imports and find_run: no data needed, so the one-cell demo below can use them on a fresh Colab
import csv, json, math, re, shutil, time
import torch
from xray3d import dataset, metrics, nets, prep, shape, torchdata, trainer


def find_run(name):
    """(Claude's plumbing) The checkpoint file of a run, or None if it isn't here.

    Looks in trainer.runs_root(); on Colab also in MyDrive/pelvis-xray3d/runs/<name>/ (Drive must be mounted),
    copying best.pt / last.pt from there. A run trained on Kaggle: download runs/<name>/ from the Kaggle
    notebook's Output tab and put it in trainer.runs_root() (or in that Drive folder for Colab).
    """
    p = Path(name)
    if p.suffix == ".pt":
        return p if p.exists() else None
    rdir = trainer.run_dir(name)
    droot = trainer.drive_runs_root()
    if droot is not None and not (rdir / f"{WHICH}.pt").exists() and (droot / name / f"{WHICH}.pt").exists():
        rdir.mkdir(parents=True, exist_ok=True)
        for f in (droot / name).glob("*"):
            if f.is_file():
                shutil.copy2(f, rdir / f.name)
        print("copied", name, "from Drive")
    ck = rdir / f"{WHICH}.pt"
    return ck if ck.exists() else None

In [ ]:
TEST = dataset.subjects("test")
print(f"{len(TEST)} test subjects:", " ".join(TEST))

CKPT = find_run(MODEL)
HAVE_MODEL = CKPT is not None
if HAVE_MODEL:
    info = trainer.load_checkpoint(CKPT)
    CFG = info["config"]
    N_VIEWS = int(CFG["n_views"])
    model = trainer.load_model(CKPT)
    print(f"model {MODEL!r} ({WHICH}.pt): {N_VIEWS} view(s), step {info['step']}, seed {info['seed']}, "
          f"best val Dice {info.get('best_val_dice', float('nan')):.3f}, train_frac {CFG.get('train_frac', 1.0)}")
    if info["step"] < 1000:
        print(f"\n*** WARNING: this model trained for only {info['step']} steps. Every number it produces below is "
              "MEANINGLESS (a smoke test of the code, not a result). Use a real run for the report. ***")
else:
    print(f"No checkpoint for {MODEL!r} in {trainer.runs_root()}. Part A's TODOs and checks still run; the demo cells "
          "will say they're skipped. Get the run from Kaggle (see the text above) and rerun from the top.")

---
## Part A · Two X-rays → a mesh (after video 14, the fly-through)

Until now the network was always fed by `torchdata` (Claude's) and scored by `training.validation_dice` on whole
batches. A user of the method has something else: **one** patient's two images, and wants a 3D bone in millimetres.
Three small functions, each one an interface you have to get exactly right.

### TODO A1 · `prepare_inputs`  ·  suggested driver: **Adir**

The **network's input contract** (units 04–08): a float32 tensor `(B, V, 128, 128)`, channel 0 = AP, channel 1 = LAT,
each image **standardised on its own** (zero mean, unit std) with `torchdata.standardise`, the same function the
network was trained with. One subject is a batch of one. Anything else (a wrong size, a missing LAT for a 2-view
model) should stop here with a clear `ValueError`, not somewhere deep inside the network.

> **Nudge:** one image at a time: check its shape, standardise it; then stack the views in the right order and add the batch axis. `n_views=None` means 'as many as I was given'.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def prepare_inputs(ap, lat=None, n_views=None):
    """The network's input for one subject.

    ap, lat : (128, 128) images (NumPy or torch), the raw line integrals as stored in the dataset; lat may be None
    n_views : 1 (AP only, even if lat is given) or 2 (AP and LAT); None = 1 if lat is None else 2
    returns : torch.float32 tensor (1, n_views, 128, 128), every image standardised on its own
    raises  : ValueError for a wrong image shape, n_views not 1/2, or n_views=2 without a LAT image
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_prepare_inputs(prepare_inputs)

In [ ]:
d = dataset.load_subject(TEST[0])
x = prepare_inputs(d["ap"], d["lat"])
plotting.show_row([d["ap"], d["lat"], x[0, 0].numpy(), x[0, 1].numpy()],
                  [f"{TEST[0]} AP, raw p", "LAT, raw p", "AP as the network sees it", "LAT as the network sees it"], size=3.2)
print(f"raw AP: {d['ap'].min():.1f} .. {d['ap'].max():.1f};  network input: mean {float(x.mean()):+.1e}, std {float(x.std()):.3f}")

### TODO A2 · `predict_occupancy`  ·  suggested driver: **Noa**

Logits → probability → mask. The network outputs logits `(B, 1, 64, 64, 64)`; the probability is their sigmoid,
the occupancy is probability **> threshold** (0.5 by the contract). Prediction is not training: eval mode (dropout
off) and no gradients. Return NumPy arrays, because everything after this (metrics, meshes, plots) is NumPy.

> **Nudge:** the same two rules as `validation_dice` in unit 05 (which mode, which context manager). Then drop the channel axis, move to the CPU, and threshold the *probability*, so `threshold` means what it says.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def predict_occupancy(model, x, threshold=0.5):
    """Run the network on a prepared batch.

    model     : a trained XrayTo3D (any nn.Module with parameters, returning (B, 1, D, H, W) logits)
    x         : (B, V, 128, 128) tensor from prepare_inputs (move it to the model's device)
    threshold : a voxel is bone if its probability is > threshold
    returns   : (prob, occ): NumPy arrays of shape (B, D, H, W); prob float32 in [0, 1], occ uint8 0/1
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_predict_occupancy(predict_occupancy)

### TODO A3 · `predict_from_xrays`  ·  suggested driver: **Adir**

The whole pipeline in one call: the two X-rays of one patient in, the occupancy, the probabilities and a **mesh in
millimetres** out (unit 02's `shape.mesh_from_occupancy`, in the cube frame: origin at the cube centre, RAS axes).
`lat=None` is the 1-view model. This is the function the demo and the report figures use.

> **Nudge:** three lines of glue, and one question per line: what shape does each step hand to the next (a batch of one in, one cube out)? Where do the millimetres come from?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def predict_from_xrays(model, ap, lat=None, threshold=0.5, cell_mm=5.0):
    """Two X-rays (or one) -> the predicted pelvis.

    model   : a trained XrayTo3D with as many views as images given
    ap, lat : (128, 128) raw images (lat=None for a 1-view model)
    returns : {"occ": (64, 64, 64) uint8, "prob": (64, 64, 64) float32,
               "verts": (N, 3) mm in the cube frame, "faces": (M, 3) int}
    """
    raise NotImplementedError("TODO A3")

In [ ]:
T.check_predict_from_xrays(predict_from_xrays)

**Explain-back (part A):**
1. Why standardise each image on its own and not the pair together? What would a real X-ray from a brighter
   machine do to the network's input otherwise? (Unit 07 standardises its real frames the same way.)
2. A teammate forgets `model.eval()`. With our network, what changes, and how would you notice? Without
   `torch.no_grad()`?
3. `prob > 0.5` and `logit > 0` give the same mask. When would you pick a threshold other than 0.5, and on which split
   would you choose it?
4. The mesh is in mm with its origin at the cube centre. Where is the patient's left femoral head in these
   coordinates (sign of x)? What would you need to place the mesh back in the CT scanner's coordinates?

**Part A done.** Move `prepare_inputs`, `predict_occupancy` and `predict_from_xrays` into `src/xray3d/pipeline.py`
(with `import torch` and `from xray3d import torchdata, shape`).

---
## The demo (Claude's cells, your functions)

First the trained network on **every** test subject, scored with your `metrics.evaluate` and the CE angle of
unit 02 (from the GT femoral heads, on both cubes). Then one subject, shown the way it'll be shown in class.

In [ ]:
def ce_angles(occ, heads):
    """Unit 02's CE angle on both sides (NaN where the femoral head is missing)."""
    out = {}
    for side, h in zip(("left", "right"), heads):
        ok = np.all(np.isfinite(h))
        out[side] = metrics.ce_angle(occ, h[:3], h[3], side) if ok else float("nan")
    return out


NET_ROWS, PRED = [], {}
if HAVE_MODEL:
    t0 = time.time()
    for sid in TEST:
        d = dataset.load_subject(sid)
        out = predict_from_xrays(model, d["ap"], d["lat"] if N_VIEWS == 2 else None)
        PRED[sid] = out
        row = {"method": f"net:{MODEL}", "subject": sid, **metrics.evaluate(d["occ"], out["occ"])}
        gt_ce, pr_ce = ce_angles(d["occ"], d["femoral_heads"]), ce_angles(out["occ"], d["femoral_heads"])
        for side in ("left", "right"):
            row[f"ce_{side}_gt"], row[f"ce_{side}_pred"] = gt_ce[side], pr_ce[side]
        NET_ROWS.append(row)
    print(f"{len(TEST)} test subjects in {time.time() - t0:.0f} s\n")
    print(f"{'subject':<9}{'Dice':>7}{'HD95 mm':>9}{'ASSD mm':>9}{'NSD 5mm':>9}{'CE L gt/pred':>16}{'CE R gt/pred':>16}")
    for r in NET_ROWS:
        print(f"{r['subject']:<9}{r['dice']:>7.3f}{r['hd95']:>9.1f}{r['assd']:>9.2f}{r['nsd_5mm']:>9.3f}"
              f"{r['ce_left_gt']:>9.1f}/{r['ce_left_pred']:<6.1f}{r['ce_right_gt']:>9.1f}/{r['ce_right_pred']:<6.1f}")
    dices = np.array([r["dice"] for r in NET_ROWS])
    print(f"\nmean Dice {dices.mean():.3f} ± {dices.std(ddof=1):.3f} (sd over {len(dices)} subjects)")
    if DEMO_SUBJECT is None:   # the median subject: a typical case, not a cherry-picked one
        DEMO_SUBJECT = NET_ROWS[int(np.argsort(dices)[len(dices) // 2])]["subject"]
    print("demo subject:", DEMO_SUBJECT)
else:
    print("skipped: no model (see the parameters cell)")

In [ ]:
if HAVE_MODEL:
    sid = DEMO_SUBJECT
    d, out = dataset.load_subject(sid), PRED[sid]
    row = next(r for r in NET_ROWS if r["subject"] == sid)
    gt, pr = d["occ"].astype(bool), out["occ"].astype(bool)
    j = int(np.argmax(gt.sum(axis=(0, 2))))                       # the coronal layer with the most bone
    fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
    plotting.show(d["ap"], f"{sid}: AP (input 1)", ax=ax[0])
    plotting.show(d["lat"], "LAT (input 2)" if N_VIEWS == 2 else "LAT (not used: 1-view model)", ax=ax[1])
    ax[2].imshow(prep.overlap_image(prep.coronal(gt, j), prep.coronal(pr, j))); ax[2].set_title(f"coronal layer j={j}"); ax[2].axis("off")
    k = int(np.argmax(gt.sum(axis=(0, 1))))
    ax[3].imshow(prep.overlap_image(gt[:, :, k].T[::-1, ::-1], pr[:, :, k].T[::-1, ::-1])); ax[3].set_title(f"axial layer k={k} (anterior up)"); ax[3].axis("off")
    fig.suptitle("white: both · red: predicted, not in the CT · blue: in the CT, missed  (TotalSegmentator, CC BY 4.0)")
    plt.tight_layout(); plt.show()

    gv, gf = shape.mesh_from_occupancy(d["occ"])
    fig = plt.figure(figsize=(13, 6))
    for i, (v, f, t, c) in enumerate([(out["verts"], out["faces"], f"predicted ({MODEL})", "#e8dcc0"),
                                      (gv, gf, "ground truth (the CT's mask)", "#b8d0e8")]):
        prep.show_mesh(v, f, t, ax=fig.add_subplot(1, 2, i + 1, projection="3d"), color=c, view=(10, -60))
    fig.suptitle(f"{sid}: Dice {row['dice']:.3f} · HD95 {row['hd95']:.1f} mm · ASSD {row['assd']:.2f} mm · "
                 f"NSD@5mm {row['nsd_5mm']:.3f} · Chamfer {row['chamfer']:.2f} mm")
    plt.tight_layout(); plt.show()
    print("CE angle (deg)  left: CT {:.1f}, predicted {:.1f}   right: CT {:.1f}, predicted {:.1f}".format(
        row["ce_left_gt"], row["ce_left_pred"], row["ce_right_gt"], row["ce_right_pred"]))

**Export.** The two meshes as `.obj` files (open them in Windows 3D Viewer, MeshLab or Blender). They go to
`env.processed_root() / "demo"` (never in git: derived from the CTs, and large-ish).

In [ ]:
if HAVE_MODEL:
    demo_dir = (Path("/kaggle/working") if env.where() == "kaggle" else env.processed_root()) / "demo"
    safe = Path(str(MODEL)).stem
    for tag, (v, f) in {"pred": (out["verts"], out["faces"]), "gt": (gv, gf)}.items():
        p = prep.write_obj(demo_dir / f"{sid}_{safe}_{tag}.obj", v, f)
        print(f"wrote {p}  ({len(v):,} vertices, {len(f):,} triangles)")

### For class: the one-cell demo (Colab)

Open this notebook on Colab, run the setup cell, the parameters cell and the cell after it (imports and `find_run`;
it needs no data), then this one. It mounts Drive, unzips the
processed dataset (`dataset.ensure_local()`), fetches the run from `MyDrive/pelvis-xray3d/runs/<MODEL>/` and shows one
test patient with an interactive 3D view (drag to rotate). It uses the functions in `src/xray3d/pipeline.py`, so
part A must be moved and pushed first. Change `CLASS_SUBJECT` to show another patient.

In [ ]:
from xray3d import dataset, metrics, prep, shape, trainer   # all this cell needs (with the setup cell and find_run)

CLASS_SUBJECT = DEMO_SUBJECT      # or any id from dataset.subjects("test")

if env.where() == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    dataset.ensure_local()
from xray3d import pipeline
_predict = getattr(pipeline, "predict_from_xrays", None) or predict_from_xrays   # yours, from src (or this notebook)
_ckpt = find_run(MODEL)
if _ckpt is None:
    print(f"no run {MODEL!r}: put runs/{MODEL}/{WHICH}.pt in MyDrive/pelvis-xray3d/runs/ (Colab) or trainer.runs_root()")
else:
    _net = trainer.load_model(_ckpt)
    _nv = int(trainer.load_checkpoint(_ckpt)["config"]["n_views"])
    CLASS_SUBJECT = CLASS_SUBJECT or dataset.subjects("test")[0]
    _d = dataset.load_subject(CLASS_SUBJECT)
    _out = _predict(_net, _d["ap"], _d["lat"] if _nv == 2 else None)
    _m = metrics.evaluate(_d["occ"], _out["occ"])
    plotting.show_row([_d["ap"], _d["lat"]], [f"{CLASS_SUBJECT}: AP", "LAT"], size=3.5)
    _gv, _gf = shape.mesh_from_occupancy(_d["occ"])
    title = f"{CLASS_SUBJECT}: Dice {_m['dice']:.3f}, HD95 {_m['hd95']:.1f} mm (beige: predicted, blue: CT)"
    try:
        import plotly.graph_objects as go     # interactive on Colab; falls back to matplotlib without plotly
        shift = np.array([220.0, 0, 0])       # side by side: the CT's mesh 220 mm to the viewer's side
        meshes = [go.Mesh3d(x=v[:, 0] + s[0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2], color=c, flatshading=True)
                  for v, f, s, c in [(_out["verts"], _out["faces"], 0 * shift, "#e8dcc0"), (_gv, _gf, shift, "#9dbbe0")] if len(v)]
        fig = go.Figure(meshes)
        fig.update_layout(title=title, scene=dict(aspectmode="data", xaxis_title="x (R) mm", yaxis_title="y (A) mm",
                                                  zaxis_title="z (S) mm"), height=520, margin=dict(l=0, r=0, t=40, b=0))
        fig.show()
    except ImportError:
        fig = plt.figure(figsize=(12, 5.5))
        prep.show_mesh(_out["verts"], _out["faces"], "predicted", ax=fig.add_subplot(121, projection="3d"))
        prep.show_mesh(_gv, _gf, "CT", ax=fig.add_subplot(122, projection="3d"), color="#9dbbe0")
        fig.suptitle(title); plt.show()

---
## Part B · The results story (after video 14, the results)

Three files, three units, one story: **baselines** (unit 03) → **the network, 1 vs 2 views and training-set size**
(unit 06) → **simulated vs real** (unit 07). Claude's reader below loads whichever exist (one row per method and test
subject, numbers as floats) and says which are missing.

In [ ]:
def read_results(name):
    """(Claude's) rows of results/<name> as dicts, numbers converted to float; None if the file isn't there."""
    p = env.results_root() / name
    if not p.exists():
        return None
    def num(v):
        try:
            return float(v)
        except (TypeError, ValueError):
            return v
    with open(p, newline="", encoding="utf-8") as fo:
        return [{k: num(v) for k, v in r.items()} for r in csv.DictReader(fo)]


SOURCES = {"baselines.csv": "unit 03 notebook (the results-table cell)",
           "experiment_v1.csv": "unit 06 (the Kaggle runs + its analysis notebook)",
           "sim_vs_real_v1.csv": "unit 07 (aggregate numbers only: DeepFluoro is CC BY-NC-ND)"}
RES = {}
for name, who in SOURCES.items():
    RES[name] = read_results(name)
    state = f"{len(RES[name])} rows" if RES[name] is not None else "MISSING"
    print(f"{name:<22}{state:>10}   <- {who}")
BASE, EXP, SVR = (RES[n] for n in SOURCES)

### TODO B1 · `best_and_worst`  ·  suggested driver: **Noa**

A mean hides the cases that matter. Rank one method's test subjects by a metric and return the `k` best and the `k`
worst. Mind the **direction**: Dice, IoU and NSD are higher-is-better, HD95, ASSD and Chamfer are distances
(lower is better). An empty prediction has HD95 = inf (it is the worst, and sorts that way). A NaN means the
evaluation failed: it can't be ranked, so leave it out.

> **Nudge:** filter, drop the NaNs, then one `sorted(..., key=...)` with the direction chosen by the metric's name. The worst list is the same order read from the other end.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def best_and_worst(rows, k=3, metric="dice", method=None):
    """The k best and the k worst rows by one metric.

    rows   : list of dicts with at least "method", "subject" and the metric (floats), e.g. read_results(...)
    metric : "dice", "iou", "nsd_1.5mm", "nsd_5mm" (higher is better) or "hd95", "assd", "chamfer" (lower is better)
    method : keep only rows with row["method"] == method; None = all rows
    returns: (best, worst), two lists of at most k rows each: best first, worst first. NaN rows are left out.
    """
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_best_and_worst(best_and_worst)

In [ ]:
# the network's 2 best and 2 worst test subjects (or, without a model, unit 03's nearest neighbour)
rows_bw = NET_ROWS if NET_ROWS else [r for r in (BASE or []) if r["method"] == "nn_ap_lat"]
if rows_bw:
    best, worst = best_and_worst(rows_bw, k=2)
    fig, ax = plt.subplots(2, 4, figsize=(15, 7.5))
    for col, (r, tag) in enumerate([(r, "best") for r in best] + [(r, "worst") for r in worst]):
        d = dataset.load_subject(r["subject"])
        gt = d["occ"].astype(bool)
        pr = PRED[r["subject"]]["occ"].astype(bool) if r["subject"] in PRED else None
        plotting.show(d["ap"], f"{tag}: {r['subject']}  Dice {r['dice']:.3f}, HD95 {r['hd95']:.0f} mm", ax=ax[0, col])
        if pr is not None:
            ax[1, col].imshow(prep.overlap_image(prep.coronal(gt), prep.coronal(pr, int(np.argmax(gt.sum(axis=(0, 2)))))))
        else:
            ax[1, col].imshow(prep.coronal(gt), cmap="gray")
        ax[1, col].axis("off")
    fig.suptitle(f"{rows_bw[0]['method']}: best and worst test subjects (coronal: white both, red extra, blue missed)")
    plt.tight_layout(); plt.show()
else:
    print("nothing to rank yet: no model and no results/baselines.csv")

**Look before you explain.** What do the worst cases have in common: a big or small pelvis, a tilted one, a
different body size in the X-ray, a part cut off? Is the worst case's failure a wrong *size*, a wrong *position* or
a wrong *shape*? (Unit 07's error analysis asks the same about real X-rays.)

### TODO B2 · `story_table`  ·  suggested driver: **Adir**

One table for the whole project, honest by construction. For each group of rows (a baseline, a network setting, a
real-X-ray condition): the **mean ± standard deviation over test subjects, and n**. Three traps:
(1) unit 06 has 3 seeds per setting, so a subject appears 3 times; 27 rows are still **9 patients**. Average each
subject over its rows first, then over subjects, and report n = subjects. (2) The sd of a sample uses n − 1.
(3) An empty prediction has HD95 = inf: leave it out of the mean but **report** how many values were used
(`<metric>_n`), so the reader sees it.

> **Nudge:** group the rows by subject (a dict of lists), one mean per subject, keep the finite ones, then mean and `std(ddof=1)` (NaN with fewer than 2 values).

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def story_table(groups, metrics=("dice", "hd95")):
    """Summary of several groups of per-subject rows.

    groups  : {label: list of row dicts with "subject" and the metrics}, e.g. {"mean shape": [...], "net, 2 views": [...]}
    metrics : the metric columns to summarise
    returns : a list with one dict per group, in the groups' order:
              {"label", "n_subjects", "n_rows", and for every metric m: m_mean, m_sd, m_n}
              where each subject's value is the mean of its rows, m_mean / m_sd are over the subjects whose value is
              finite (sd with n - 1; NaN if fewer than 2), and m_n is how many subjects that was.
    """
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_story_table(story_table)

Claude's cell below fills your `story_table` with everything that exists: each baseline; the network settings of
unit 06 (or, until that file exists, this notebook's `MODEL` as a single run); unit 07's real-X-ray numbers. Unit 07's
file holds **aggregates only** (no per-subject rows may be committed for DeepFluoro), so its lines are copied in as
they are. The output is a Markdown table to paste into the report.

In [ ]:
STORY_METRICS = ("dice", "hd95", "assd", "nsd_5mm")
BASELINE_LABELS = {"mean_shape": "mean shape (same answer for all)", "nn_ap": "nearest neighbour, AP",
                   "nn_ap_lat": "nearest neighbour, AP + LAT", "hull_oracle_sil": "visual hull (oracle silhouettes)",
                   "oracle_retrieval": "oracle retrieval (uses the test GT: a ceiling)"}
groups = {}
for r in BASE or []:
    groups.setdefault("baseline: " + BASELINE_LABELS.get(r["method"], r["method"]), []).append(r)
if EXP:
    seen = set()
    for r in EXP:
        if "n_views" in r and "train_frac" in r:
            nv, tf = int(r["n_views"]), float(r["train_frac"])
            label = f"network: {nv} view{'s' if nv > 1 else ''}, {tf:.0%} of train"
        else:
            label = f"network: {r['method']}"
        key = (label, r["subject"], r.get("seed"))
        if key in seen:        # the 2-view / 100 % setting belongs to both arms: count its runs once
            continue
        seen.add(key)
        groups.setdefault(label, []).append(r)
elif NET_ROWS:
    groups[f"network: {MODEL} (this notebook, 1 run)"] = NET_ROWS
# unit 07's file is long: one row per (section, what, n_views, metric) with n, mean, sd (aggregates over the
# DeepFluoro specimens, never per specimen). Its "deepfluoro_mean" rows are the mean score of each input of the
# ladder (what = sim_full / sim_cut / drr_pose / real). Pivot them into one dict per (n_views, input) shaped like
# a story_table row (label, n_subjects, <metric>_mean / _sd / _n), so the table code below prints them too.
SVR_AGG = {}
for r in SVR or []:
    if r.get("section") != "deepfluoro_mean":
        continue
    nv = int(r["n_views"])
    a = SVR_AGG.setdefault((nv, r["what"]), {"label": f"{nv} view{'s' if nv > 1 else ''}, {r['what']} AP"
                                                   + (" + sim LAT" if nv == 2 else ""), "n_subjects": int(r["n"])})
    a[f"{r['metric']}_mean"], a[f"{r['metric']}_sd"], a[f"{r['metric']}_n"] = r["mean"], r["sd"], int(r["n"])
SVR_AGG = list(SVR_AGG.values())
# unit 07 writes smoke_models_meaningless = True when it ran on its tiny CPU models (read back as a string)
if SVR and any(str(r.get("smoke_models_meaningless")).strip().lower() in ("true", "1", "1.0") for r in SVR):
    print("*** sim_vs_real_v1.csv was written with SMOKE models: its DeepFluoro rows are MEANINGLESS. Rerun unit 07 "
          "with unit 06's runs. ***\n")

TABLE = story_table(groups, STORY_METRICS)


def cell(r, m):
    """'0.612 ± 0.045' (+ the n if some subjects were left out as non-finite)."""
    mu, sd, n = r.get(f"{m}_mean"), r.get(f"{m}_sd"), r.get(f"{m}_n")
    if mu is None or (isinstance(mu, float) and math.isnan(mu)):
        return "–"
    fmt = "{:.3f}" if m in ("dice", "iou", "nsd_1.5mm", "nsd_5mm") else "{:.1f}"
    s = fmt.format(mu) + ("" if sd is None or (isinstance(sd, float) and math.isnan(sd)) else " ± " + fmt.format(sd))
    if n is not None and "n_subjects" in r and int(n) < int(r["n_subjects"]):
        s += f" (n={int(n)})"
    return s


head = ["", "n subjects", "rows"] + [f"{m} (mean ± sd)" for m in STORY_METRICS]
lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
for r in TABLE:
    lines.append(f"| {r['label']} | {r['n_subjects']} | {r['n_rows']} | " + " | ".join(cell(r, m) for m in STORY_METRICS) + " |")
for r in SVR_AGG:      # unit 07's aggregates (n = DeepFluoro specimens), pivoted above
    label = r.get("label") or r.get("method") or r.get("condition") or "?"
    lines.append(f"| DeepFluoro: {label} | {int(r['n_subjects']) if 'n_subjects' in r else r.get('n', '?')} | – | "
                 + " | ".join(cell(r, m) for m in STORY_METRICS) + " |")
STORY_MD = "\n".join(lines)
print(STORY_MD)
missing = [n for n in SOURCES if RES[n] is None]
if missing:
    print("\nnot in the table yet (file missing):", ", ".join(missing))
if NET_ROWS and not EXP and trainer.load_checkpoint(CKPT)["step"] < 1000:
    print(f"\n*** the network row comes from {MODEL!r}, a {trainer.load_checkpoint(CKPT)['step']}-step smoke model: MEANINGLESS ***")

In [ ]:
# the same groups as one dot per subject (the figure for the report; the subject mean of each group)
show = [(r["label"], groups[r["label"]]) for r in TABLE if groups.get(r["label"])]
if show:
    fig, ax = plt.subplots(1, 2, figsize=(14, 0.55 * len(show) + 1.8))
    for a, m, label in [(ax[0], "dice", "Dice (higher is better)"), (ax[1], "hd95", "HD95, mm (lower is better)")]:
        for y, (name, rows) in enumerate(show):
            per = {}
            for r in rows:
                per.setdefault(r["subject"], []).append(float(r[m]))
            v = np.array([np.mean(x) for x in per.values()])
            v = v[np.isfinite(v)]
            a.scatter(v, np.full(len(v), y) + np.linspace(-0.15, 0.15, len(v)), s=20, color="#3b6fb6", alpha=0.7)
            if len(v):
                a.plot([v.mean()] * 2, [y - 0.3, y + 0.3], color="#1d1d1d", lw=2)
        a.set_yticks(range(len(show)), [s[0] for s in show]); a.invert_yaxis(); a.set_xlabel(label); a.grid(axis="x", alpha=0.3)
    fig.suptitle("test subjects: one dot per subject (mean over seeds), the bar is the mean")
    plt.tight_layout(); plt.show()

**Explain-back (part B):**
1. Is the network better than the **mean shape**? Than nearest neighbour? Do the mean ± sd ranges overlap, and is
   "overlap" the right test with 9 paired subjects? (Unit 06's paired comparison is.)
2. Why is n the number of subjects and not the number of rows? What would a reader conclude from "n = 27"?
3. Which **one** number would you put in the abstract, and which row would you put right next to it so the
   number can't mislead?
4. Where is the oracle row, and why must it be labelled a ceiling and not a method?

**Part B done.** Move `best_and_worst` and `story_table` into `src/xray3d/pipeline.py`.

---
## Part C · The report (after video 14, the report)

The lecturer asks for a short report **with a licence section**, a **reproducible notebook** and the
**documentation of personal contribution**. The outline is in `report_outline.md` (this folder); the report is
yours to write. Two of Claude's cells help: where every number comes from, and who did what.

### Reproduce everything

Every number in the report must come from a file below, and every file from one notebook or run. ✔ = here now.

In [ ]:
import platform, subprocess


def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=env.repo_root(), capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        return ""


R, PR = env.results_root(), env.processed_root()
CHECKLIST = [
    # (what, made by, file)
    ("the patient-level split (train / val / test)", "unit 01, part E: split_subjects (once, never regenerated)", env.repo_root() / "splits" / "split_v1.json"),
    ("processed dataset v1 (64³ cubes + DRR pairs)", "unit 02: build_dataset.ipynb (local subset or the Kaggle full build)", dataset.root() / "index.json"),
    ("baselines table (hull, NN, mean shape, oracle)", "unit 03: notebook.ipynb, the results-table cell", R / "baselines.csv"),
    ("first training run", "unit 05, part C: trainer.train('first-2view', 4000 steps)", trainer.run_dir("first-2view") / "best.pt"),
    ("1 vs 2 views, 25 % vs 100 % train (3 seeds each)", "unit 06: the Kaggle runs (units/06-experiment/run_experiment.py) + unit 06's notebook", R / "experiment_v1.csv"),
    ("the demo / report model", f"the run {MODEL!r} (unit 06's best 2-view run on val)", CKPT if CKPT is not None else trainer.run_dir(str(MODEL)) / f"{WHICH}.pt"),
    ("simulated vs real (DeepFluoro, aggregates only)", "unit 07: notebook.ipynb", R / "sim_vs_real_v1.csv"),
    ("demo meshes (.obj)", "unit 08: this notebook, the export cell", (Path("/kaggle/working") if env.where() == "kaggle" else PR) / "demo"),
]


def short(p):
    """The path from the folder that names it: results/x.csv, runs/<name>/best.pt, splits/..., processed/...."""
    for base, tag in [(R, "results"), (trainer.runs_root(), "runs"), (PR, "processed"), (env.repo_root(), "")]:
        try:
            return (Path(tag) / Path(p).relative_to(base)).as_posix()
        except ValueError:
            pass
    return str(p)


print(f"{'':2}{'what':<52}{'file':<40}made by")
for what, who, p in CHECKLIST:
    p = Path(p)
    ok = p.exists()
    extra = ""
    if ok and p.suffix == ".csv":
        extra = f"  [{sum(1 for _ in open(p, encoding='utf-8')) - 1} rows, {time.strftime('%Y-%m-%d', time.localtime(p.stat().st_mtime))}]"
    elif ok and p.is_dir():
        extra = f"  [{len(list(p.glob('*.obj')))} files]"
    print(f"{'✔' if ok else '✗':<2}{what:<52}{short(p):<40}{who}{extra}")

idx = dataset.load_index() if dataset.exists() else {}
print(f"\ncode: commit {git('rev-parse', '--short', 'HEAD') or '?'}"
      f"{' + uncommitted changes' if git('status', '--porcelain') else ''};  dataset: {len(idx.get('subjects', []))} subjects, "
      f"built {idx.get('created', '?')} from {idx.get('source', '?')}")
print(f"python {platform.python_version()}, numpy {np.__version__}, torch {torch.__version__}")
g = dataset.geometry()
print(f"geometry: source-detector {g['sdd_mm']} mm, source-isocentre {g['sid_mm']} mm, {g['detector_px']}² px of {g['pixel_mm']} mm; "
      f"cube {g['cube_n']}³ x {g['cube_mm'] / g['cube_n']} mm")
print("folders: results", R, "| runs", trainer.runs_root(), "| processed", PR)

Put this cell's output (with a ✔ on every line) in the report's appendix. If a line is ✗, the number it stands for
isn't reproducible yet: rerun the notebook named on that line, don't type the number in by hand.

### Personal contribution, from the logs

The lecturer requires the contribution of each partner to be documented. Yours is already written down: every
unit's `LOG.md` has a TODO table (driver, who explained it back, the hints used). Claude's cell below reads all of
them and counts. Write the hints column as **tier then name**, e.g. `2 Noa; 3 Adir` (as in the column's header).

In [ ]:
PEOPLE = ("Adir", "Noa")
BOTH_WORDS = ("both", "together", "יחד")


def people_in(text):
    """Which partners a cell names ('both' / 'together' = all of them)."""
    t = text.lower()
    if any(w in t for w in BOTH_WORDS):
        return set(PEOPLE)
    return {p for p in PEOPLE if re.search(rf"\b{p.lower()}\b", t)}


def hints_in(text):
    """[(tier, person or '?'), ...] from a hints cell such as '2 Noa; 3 Adir' or 'tier 2, Noa'."""
    out, pending, last = [], [], None
    for tok in re.findall(r"\b(?:[23]|" + "|".join(p.lower() for p in PEOPLE) + r")\b", text.lower()):
        if tok.isdigit():
            pending.append(int(tok))
        else:
            last = next(p for p in PEOPLE if p.lower() == tok)
            out += [(t, last) for t in pending]
            pending = []
    return out + [(t, last or "?") for t in pending]


def parse_log(path):
    """(rows of every table whose header has a 'Driver' column, lines under '## Reveals') of one LOG.md."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    split = lambda s: [c.strip() for c in s.strip().strip("|").split("|")]
    rows, reveals, i, in_reveals = [], [], 0, False
    while i < len(lines):
        s = lines[i].strip()
        if s.startswith("|") and "driver" in s.lower():
            head = [h.lower() for h in split(s)]
            i += 2                                         # skip the |---|---| line
            while i < len(lines) and lines[i].strip().startswith("|"):
                rows.append(dict(zip(head, split(lines[i]))))
                i += 1
            continue
        if s.startswith("## "):
            in_reveals = s.lower().startswith("## reveals")
        elif in_reveals and s.strip("-* ").strip():
            reveals.append(s)
        i += 1
    return rows, reveals


def contribution(units_dir):
    table = []
    for log in sorted(Path(units_dir).glob("[0-9]*/LOG.md")):
        rows, reveals = parse_log(log)
        c = {"unit": log.parent.name, "todos": len(rows), "logged": 0, "reveals": len(reveals)}
        for p in PEOPLE:
            c[f"drove {p}"] = c[f"explained {p}"] = c[f"hints {p}"] = 0
        for r in rows:
            col = lambda name: next((v for k, v in r.items() if k.startswith(name)), "")
            drivers, explainers = people_in(col("driver")), people_in(col("explained"))
            c["logged"] += bool(drivers)
            for p in drivers:
                c[f"drove {p}"] += 1
            for p in explainers:
                c[f"explained {p}"] += 1
            for tier, who in hints_in(col("hints")):
                if who in PEOPLE:
                    c[f"hints {who}"] += 1
        table.append(c)
    return table


CONTRIB = contribution(env.repo_root() / "units")
cols = ["unit", "logged"] + [f"drove {p}" for p in PEOPLE] + [f"explained {p}" for p in PEOPLE] + [f"hints {p}" for p in PEOPLE] + ["reveals"]
names = {"logged": "TODOs logged"} | {f"hints {p}": f"hints 2–3 asked by {p}" for p in PEOPLE} \
        | {f"drove {p}": f"driven by {p}" for p in PEOPLE} | {f"explained {p}": f"explained back by {p}" for p in PEOPLE}
md_lines = ["| " + " | ".join(names.get(c, c) for c in cols) + " |", "|" + "---|" * len(cols)]
for c in CONTRIB:
    md_lines.append("| " + " | ".join(str(c[k]) if k != "logged" else f"{c['logged']} / {c['todos']}" for k in cols) + " |")
tot = {k: sum(c[k] for c in CONTRIB) for k in cols[1:]} | {"unit": "**total**", "todos": sum(c["todos"] for c in CONTRIB)}
md_lines.append("| " + " | ".join(str(tot[k]) if k != "logged" else f"{tot['logged']} / {tot['todos']}" for k in cols) + " |")
CONTRIB_MD = "\n".join(md_lines)
print(CONTRIB_MD)
print("\n(a TODO driven 'both' counts for both; 'TODOs logged' = rows with a driver filled in)")

Paste the table into the report's personal-contribution section, and under it, in a few sentences each, what each
of you led (the proposal's leads: Noa the data, the metrics, the hull, the training-set size and the error analysis;
Adir the DRR, the dataset build, nearest neighbour, 1 vs 2 views and the sim-vs-real gap) and what each of you
explained back. If the table has empty units, fill those logs first.

---
## Wrap-up: run the checks against `src/xray3d`

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md` (driver per TODO, hints used, the explain-back answers, the demo numbers). Then write the report
from `report_outline.md`, and ask Claude to review it like a TA would. That's the project.